# 📷 Data Augmentation: Generar Variaciones de Imágenes

Este notebook aplica transformaciones a imágenes en `fotosParaProcesar/` y guarda las versiones aumentadas en `fotosProcesadas/`

**Transformaciones que se aplicarán:**
- Rotaciones (±20°)
- Volteos horizontales y verticales
- Cambios de brillo y contraste
- Ruido gaussiano
- Transformaciones afines (shear)

In [ ]:
pip install -q pillow opencv-python albumentations matplotlib
import os
from pathlib import Path
import cv2
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np

print('✅ Librerías cargadas correctamente')

In [ ]:
# Configuración
INPUT_DIR = Path('fotosParaProcesar')
OUTPUT_DIR = Path('fotosProcesadas')
OUTPUT_DIR.mkdir(exist_ok=True)
NUM_AUGMENTATIONS = 5

# Pipeline de transformaciones con Albumentations
transform = A.Compose([
    A.Rotate(limit=20, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.GaussNoise(p=0.3),
    A.Affine(shear=(-10, 10), p=0.5),
])

print(f'📁 Directorio entrada: {INPUT_DIR}')
print(f'📁 Directorio salida: {OUTPUT_DIR}')
print(f'🔄 Augmentaciones por imagen: {NUM_AUGMENTATIONS}')

In [ ]:
# Procesar imágenes
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_files = sorted([f for f in INPUT_DIR.rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS])

print(f'📷 Imágenes encontradas: {len(image_files)}')
print('-' * 50)

total_generated = 0

for img_path in image_files:
    print(f'🔄 Procesando: {img_path.name}')
    
    # Leer imagen
    image = cv2.imread(str(img_path))
    if image is None:
        print(f'  ⚠️ No se pudo leer: {img_path}')
        continue
    
    # Convertir BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    stem = img_path.stem
    
    # Guardar original
    orig_out = OUTPUT_DIR / f'{stem}_original.jpg'
    cv2.imwrite(str(orig_out), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    total_generated += 1
    print(f'  ✓ Original guardado')
    
    # Generar augmentaciones
    for i in range(NUM_AUGMENTATIONS):
        augmented = transform(image=image)['image']
        out_path = OUTPUT_DIR / f'{stem}_aug_{i+1}.jpg'
        aug_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(out_path), aug_bgr)
        total_generated += 1
        print(f'  ✓ Variante {i+1}/{NUM_AUGMENTATIONS}')
    
    print()

print('=' * 50)
print(f'✅ Proceso completado')
print(f'📊 Total de imágenes generadas: {total_generated}')
print(f'💾 Ubicación: {OUTPUT_DIR.absolute()}')

In [ ]:
# Visualizar ejemplos de augmentation (si existen imágenes)
output_files = list(OUTPUT_DIR.glob('*_original.jpg'))

if output_files:
    # Tomar la primera imagen como ejemplo
    sample_stem = output_files[0].stem.replace('_original', '')
    
    # Cargar original + 3 aumentaciones
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(f'Ejemplo de Augmentation: {sample_stem}', fontsize=14, fontweight='bold')
    
    # Original
    orig = cv2.imread(str(OUTPUT_DIR / f'{sample_stem}_original.jpg'))
    orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    axes[0, 0].imshow(orig_rgb)
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    # 5 variantes
    for i in range(5):
        aug_path = OUTPUT_DIR / f'{sample_stem}_aug_{i+1}.jpg'
        if aug_path.exists():
            aug = cv2.imread(str(aug_path))
            aug_rgb = cv2.cvtColor(aug, cv2.COLOR_BGR2RGB)
            row = (i + 1) // 3
            col = (i + 1) % 3
            axes[row, col].imshow(aug_rgb)
            axes[row, col].set_title(f'Variante {i+1}')
            axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No hay imágenes en fotosProcesadas/ aún. Ejecuta la celda anterior primero.')

In [ ]:
## 📋 Instrucciones de Uso

### Paso 1: Preparar imágenes
Coloca las imágenes originales de frutas en la carpeta `fotosParaProcesar/`

### Paso 2: Ejecutar augmentation
Ejecuta las celdas anteriores en orden. El notebook:
- Lee todas las imágenes de `fotosParaProcesar/`
- Guarda la original en `fotosProcesadas/`
- Genera 5 variaciones de cada imagen con transformaciones aleatorias

### Paso 3: Organizar para entrenamiento
Una vez satisfecho con las augmentaciones:
1. Copia las imágenes a las carpetas de clase:
   - `data/bueno/`
   - `data/regular/`
   - `data/malo/`
2. Ejecuta el notebook `taller_calidad_frutas_bueno_regular_malo.ipynb` para entrenar

### Parámetros disponibles
- `NUM_AUGMENTATIONS`: Cambiar cantidad de variaciones por imagen (línea 5 de celda 3)

In [ ]:
# Estadísticas finales
all_files = list(OUTPUT_DIR.glob('*.jpg'))
original_count = len([f for f in all_files if '_original' in f.name])
augmented_count = len([f for f in all_files if '_aug_' in f.name])

print('📊 ESTADÍSTICAS FINALES')
print('=' * 50)
print(f'Imágenes originales: {original_count}')
print(f'Imágenes aumentadas: {augmented_count}')
print(f'Total de archivos JPG: {len(all_files)}')
print(f'Espacio en disco: {sum(f.stat().st_size for f in all_files) / (1024*1024):.2f} MB')
print('=' * 50)
print('✅ Listo para entrenar el modelo')